# EDA & Feature Engineering — Jena Climate Dataset
**Task:** Multivariate time-series regression  
**Target:** `T (degC)` (air temperature)

In [ ]:
import sys
from pathlib import Path

# Make the utils package importable from the notebook
sys.path.insert(0, str(Path('..').resolve()))

import pandas as pd
import matplotlib.pyplot as plt

from utils.loader import load_data, inspect
from utils.time_features import parse_timestamps, extract_time_features, resample_hourly
from utils.visualizer import (
    plot_correlation_heatmap,
    plot_histograms,
    plot_time_series,
    plot_temperature_heatmap,
)
from utils.cleaner import replace_sentinels, interpolate_missing, handle_outliers
from utils.features import add_lag_features, add_rolling_features, add_time_of_day, drop_lag_nans

DATA_PATH = Path('../data/jena_climate_2009_2016.csv')
OUTPUT_PATH = Path('../output/processed_data.csv')
OUTPUT_PATH.parent.mkdir(exist_ok=True)

TARGET = 'T (degC)'
%matplotlib inline
plt.rcParams['figure.dpi'] = 100

---
## Section 1 — Data Loading & Initial Inspection

In [ ]:
raw = load_data(DATA_PATH)
shape_raw = raw.shape
print(f'Raw shape: {shape_raw}')

In [ ]:
inspect(raw)

---
## Section 2 — Timestamp Handling

In [ ]:
# Parse the Date Time column and set as index
df = parse_timestamps(raw)
print(f'Date range: {df.index.min()} → {df.index.max()}')
print(f'Shape after parsing: {df.shape}')

In [ ]:
# Resample to hourly (raw data is every 10 min)
df_h = resample_hourly(df)
print(f'Hourly-resampled shape: {df_h.shape}')

In [ ]:
# Extract time features from the hourly index
df_h = extract_time_features(df_h)
print('Time feature columns added:', ['hour', 'day_of_week', 'month', 'is_weekend'])
df_h[['hour', 'day_of_week', 'month', 'is_weekend']].head()

---
## Section 3 — Visualization & Distribution Analysis

In [ ]:
# Correlation heatmap — note top correlators: Tdew, rh, Tpot
plot_correlation_heatmap(df_h, target=TARGET)

In [ ]:
# Histograms — look for suspicious spikes (e.g. wv has -9999 sentinels before cleaning)
plot_histograms(df_h)

In [ ]:
# Time series: raw T + rolling means (daily & weekly cycles visible)
plot_time_series(df_h, col=TARGET)

In [ ]:
# Mean temperature by hour-of-day × weekday
plot_temperature_heatmap(df_h, target=TARGET)

---
## Section 4 — Cleaning: Sentinel Values, Missing Data & Outliers

In [ ]:
# Step 4a — Replace -9999 sentinels in wind speed / direction columns with NaN
df_clean = replace_sentinels(df_h)

# Verify sentinels are gone
print('Remaining -9999 values:', (df_clean == -9999).sum().sum())

In [ ]:
# Step 4b — Time-based interpolation for remaining NaNs
# Rationale: linear interpolation respects the temporal ordering of the series
df_clean = interpolate_missing(df_clean)

In [ ]:
# Step 4c — Cap physical outliers using domain knowledge
# Rationale: values outside the physical plausible range are measurement errors;
# capping (not dropping) preserves temporal continuity of the time series.
df_clean, outlier_report = handle_outliers(df_clean, method='physical')
print('\nOutlier report:', outlier_report)

---
## Section 5 — Feature Engineering

In [ ]:
# Lag features: T at 1h, 2h, 3h ago
df_feat = add_lag_features(df_clean, col=TARGET, lags=[1, 2, 3])
print('Lag columns added:', [c for c in df_feat.columns if 'lag' in c])

In [ ]:
# Rolling mean features: 3 h, 6 h, 12 h windows (hourly-resampled data)
df_feat = add_rolling_features(df_feat, col=TARGET, windows=[3, 6, 12])
print('Rolling columns added:', [c for c in df_feat.columns if 'rolling' in c])

In [ ]:
# Time-of-day category (night / morning / afternoon / evening) + ordinal encoding
df_feat = add_time_of_day(df_feat)
print(df_feat['time_of_day'].value_counts())

In [ ]:
# Drop rows with NaN introduced by lag features (first N rows)
df_feat = drop_lag_nans(df_feat)

---
## Section 6 — Save & Report

In [ ]:
df_feat.to_csv(OUTPUT_PATH)
print(f'Saved processed data to: {OUTPUT_PATH.resolve()}')
print(f'Final shape: {df_feat.shape}')
print(f'Columns ({len(df_feat.columns)}):', df_feat.columns.tolist())

In [ ]:
# ── Final Summary ──────────────────────────────────────────────────────────────
summary = f"""
FINDINGS SUMMARY
================
• Raw shape          : {shape_raw}
• After hourly resample : {df_h.shape}
• After cleaning     : {df_clean.shape}
• Final (with features): {df_feat.shape}

• Top correlations with {TARGET}:
    Tdew (degC) ~0.99, Tpot (K) ~1.00, VPmax ~0.95, rh (%) ~-0.56

• Sentinel values replaced: -9999 in wv (m/s), max. wv (m/s), wd (deg)

• Outlier handling: physical bounds capping per column
  (see outlier_report above for column-level counts)

• New feature columns:
    Time features : hour, day_of_week, month, is_weekend
    Lag features  : T_lag_1h, T_lag_2h, T_lag_3h
    Rolling means : T_rolling_3, T_rolling_6, T_rolling_12
    Time-of-day   : time_of_day (string), time_of_day_ord (int 0-3)
"""
print(summary)